# 17 — Tucker Clustering, Fixed (Partial Tucker on patient axis)

**Bug found in notebook 15:** `tucker(tensor, rank=(416, 1000, 3))` was called with the
patient mode (mode 0) rank set to its **full size (416)**, intending that to mean
"don't touch the patient axis." That assumption is wrong. Verified directly (toy example):
even at full rank, Tucker's Higher-Order Orthogonal Iteration still computes a genuine,
dense orthonormal **rotation** matrix for that mode -- not the identity. This means
`core.reshape(416, -1)` in notebooks 14/15 did **not** give a per-original-patient
representation: row k of the core is a linear combination across *all* 416 original
patients, not patient k's own data. Clustering on that representation and then comparing
the resulting labels against `cdr_series` (indexed by the real patient order) compared
two mismatched orderings -- which explains both the near-zero ARI/NMI and the single
extreme value (~510) sitting at row 0: that's just the dominant singular value of the
patient-mode unfolding (analogous to a PC1 eigenvalue), unrelated to any real patient
being an outlier (verified: the real `OAS1_0001` looks completely unremarkable in the
raw HOG data, z-scores all under 1 across every image type).

**Fix:** use `tensorly.decomposition.partial_tucker(modes=[1, 2], rank=[hog_rank, img_rank])`
instead of full `tucker`. This decomposes *only* the HOG-feature mode and the image-type
mode, leaving the patient mode completely untouched -- so `core[k, :, :]` is guaranteed
to correspond to the original patient k, in the original order. Verified on a toy tensor:
reconstruction confirms no mixing occurs across the patient axis.

Same downstream pipeline as notebook 15 for a fair comparison: HOG-rank=1000, image-rank=3,
PCA 90%, Robust Mahalanobis distance, FasterPAM k=3/4.

## Install

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-400:] if r.stderr else "(no stderr)")

for pkg in ["tensorly", "kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    tensorly
  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Imports

In [2]:
import os
import time
import numpy as np
import pandas as pd
import tensorly as tl
from tensorly.decomposition import partial_tucker
from sklearn.decomposition import PCA

import kmedoids
from robust_mixed_dist.mixed import robust_mahalanobis_dist_matrix, S_robust
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 1 — Load raw HOG features and CDR (same as notebook 15)

In [3]:
NB_FOLDER = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NB_FOLDER, '..', '..'))
DATA_PIPELINE_DIR = os.path.join(PROJECT_ROOT, 'notebooks', '01_data_pipeline')
OUTPUT_DIR = NB_FOLDER
NB_DIR = DATA_PIPELINE_DIR      # legacy alias: shared pipeline inputs (pca_*.csv, clinical xlsx)
NB_PADDED = OUTPUT_DIR          # legacy alias: this notebook's own outputs

HOG_LEN = 8_100

print("Loading raw_hog_features_per_image.csv ...", end=" ", flush=True)
df_hog      = pd.read_csv(os.path.join(NB_DIR, "raw_hog_features_per_image.csv"))
patient_ids = df_hog["patient_id"].values
X_flat      = df_hog.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)
print(f"done  {X_flat.shape}")

blocks    = [X_flat[:, i * HOG_LEN:(i + 1) * HOG_LEN] for i in range(5)]
tensor    = np.stack(blocks, axis=2)   # (416, 8100, 5)
tensor_tl = tl.tensor(tensor)
print(f"Tensor shape : {tensor.shape}")

df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})
df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)
cdr_series = df_clin.set_index("patient_id")["CDR"].reindex(patient_ids)
mask_cdr   = cdr_series.notnull().values
cdr_int    = (cdr_series.values[mask_cdr] * 2).astype(int)

print(f"Patients with CDR : {mask_cdr.sum()} / {len(patient_ids)}")

Loading raw_hog_features_per_image.csv ... done  (416, 40500)
Tensor shape : (416, 8100, 5)
Patients with CDR : 235 / 416


## Step 2 — Partial Tucker decomposition (patient axis untouched)

`modes=[1, 2]` decomposes only the HOG-feature mode (8,100 -> 1,000) and the
image-type mode (5 -> 3). Mode 0 (patients, 416) is **not** included in `modes`,
so it is left completely unfactorized -- `core[k, :, :]` corresponds exactly to
the original patient `k`, in the original order.

In [18]:
HOG_RANK = 10
IMG_RANK = 1

print(f"Partial Tucker (modes=[1,2], rank=[{HOG_RANK},{IMG_RANK}]) ...", end=" ", flush=True)
t0 = time.time()
(core, factors), rec_errors = partial_tucker(
    tensor_tl, modes=[1, 2], rank=[HOG_RANK, IMG_RANK], random_state=42
)
print(f"done in {time.time()-t0:.1f}s")

core_np     = tl.to_numpy(core)            # (416, 1000, 3) -- mode 0 untouched
tucker_repr = core_np.reshape(416, -1)     # (416, 3000) -- now a TRUE per-patient repr

print(f"Core shape         : {core_np.shape}  (expect (416, {HOG_RANK}, {IMG_RANK}))")
print(f"Tucker repr shape  : {tucker_repr.shape}")
print(f"Final recon error  : {rec_errors[-1]:.4f}")
print(f"Non-zero var cols  : {(np.var(tucker_repr, axis=0) > 1e-10).sum()} / {tucker_repr.shape[1]}")
print(f"NaN values         : {np.isnan(tucker_repr).sum()}")

# Sanity check: the dominant per-column variance should no longer be a single
# absurd outlier the way it was with full (non-partial) Tucker in notebook 14/15.
col_var = np.var(tucker_repr, axis=0)
print(f"\nPer-column variance range : [{col_var.min():.6f}, {col_var.max():.6f}]")
print(f"Variance of top column / median column ratio: {col_var.max() / np.median(col_var):.1f}x")

Partial Tucker (modes=[1,2], rank=[10,1]) ... done in 5.6s
Core shape         : (416, 10, 1)  (expect (416, 10, 1))
Tucker repr shape  : (416, 10)
Final recon error  : 0.6110
Non-zero var cols  : 10 / 10
NaN values         : 0

Per-column variance range : [0.162194, 2.234937]
Variance of top column / median column ratio: 3.0x


## Step 3 — PCA at 90%

In [5]:
PCA_THRESH = 0.90
SAFE_LIMIT = len(patient_ids) / 3   # 138.67

pca       = PCA(n_components=PCA_THRESH, random_state=42)
X_final   = pca.fit_transform(tucker_repr)
N_FINAL   = X_final.shape[1]
var_total = pca.explained_variance_ratio_.sum() * 100

print(f"Tucker repr  : {tucker_repr.shape}")
print(f"After PCA    : {X_final.shape}  ({var_total:.1f}% of Tucker variance)")
print(f"Safe limit   : < {SAFE_LIMIT:.2f}")
print(f"Status       : {'SAFE' if N_FINAL < SAFE_LIMIT else 'UNSAFE'}  ({N_FINAL} components)")
print(f"\nTop 5 explained variance ratios: {pca.explained_variance_ratio_[:5].round(4)}")

Tucker repr  : (416, 3000)
After PCA    : (416, 308)  (90.1% of Tucker variance)
Safe limit   : < 138.67
Status       : UNSAFE  (308 components)

Top 5 explained variance ratios: [0.0699 0.0342 0.0275 0.0188 0.0154]


## Step 4 — Robust Mahalanobis distance matrix

In [19]:
print(f"Computing Robust Mahalanobis on tucker_repr {tucker_repr.shape} ...", end=" ", flush=True)
S_est = S_robust(tucker_repr, method="trimmed", alpha=0.05)
D     = robust_mahalanobis_dist_matrix(tucker_repr, S_est)
print("done")
print(f"D shape      : {D.shape}")
print(f"Value range  : [{D.min():.4f}, {D.max():.4f}]")
print(f"NaN values   : {np.isnan(D).sum()}")
print(f"Is symmetric : {np.allclose(D, D.T)}")

Computing Robust Mahalanobis on tucker_repr (416, 10) ... done
D shape      : (416, 416)
Value range  : [0.0000, 10.6016]
NaN values   : 0
Is symmetric : True


## Step 5 — Cluster and evaluate (k=3 and k=4)

Same evaluation as notebook 15, for direct comparison.

In [20]:
all_results = []

print(f"{'='*65}")

print(f"{'='*65}")

for k in [3, 4]:
    result = kmedoids.fasterpam(D, medoids=k, random_state=42)
    labels = np.array(result.labels, dtype=int)

    df_eval = pd.DataFrame({"cluster": labels, "CDR": cdr_series.values})
    df_eval = df_eval[df_eval["CDR"].notnull()].copy()
    pct_tab = pd.crosstab(
        df_eval["cluster"], df_eval["CDR"], normalize="index"
    ).mul(100).round(1)
    pct_h  = pct_tab[0.0] if 0.0 in pct_tab.columns \
             else pd.Series(0.0, index=pct_tab.index)
    spread = float(pct_h.max() - pct_h.min())

    df_ev2 = df_eval.copy()
    df_ev2["CDR"] = df_ev2["CDR"].astype(str)
    ct = pd.crosstab(df_ev2["cluster"], df_ev2["CDR"],
                     margins=True, margins_name="Total")
    print(f"\n  k={k}  loss={result.loss:.4f}  "
          f"sizes={ {c: int((labels==c).sum()) for c in sorted(set(labels))} }")
    print("  Count:")
    print(ct.to_string())
    print("  Row %:")
    print(pct_tab.to_string())

    sil = silhouette_score(D, labels, metric="precomputed")
    ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
    nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

    print(f"\n  spread={spread:.2f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

    all_results.append({
        "n_final_components": tucker_repr.shape,
        "k":                  k,
        "silhouette":         round(sil, 4),
        "ari":                round(ari, 4),
        "nmi":                round(nmi, 4),
        "spread_pct_healthy": round(spread, 2),
    })

print("\n\nClustering complete.")


  k=3  loss=1273.4325  sizes={np.int64(0): 151, np.int64(1): 120, np.int64(2): 145}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         80   29   10    1    120
1         36   36   16    1     89
2         19    5    2    0     26
Total    135   70   28    2    235
  Row %:
CDR       0.0   0.5   1.0  2.0
cluster                       
0        66.7  24.2   8.3  0.8
1        40.4  40.4  18.0  1.1
2        73.1  19.2   7.7  0.0

  spread=32.70%  sil=0.0679  ARI=0.0406  NMI=0.0397

  k=4  loss=1225.2457  sizes={np.int64(0): 160, np.int64(1): 110, np.int64(2): 128, np.int64(3): 18}
  Count:
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         64   32   22    2    120
1         22   13    1    0     36
2         41   21    2    0     64
3          8    4    3    0     15
Total    135   70   28    2    235
  Row %:
CDR       0.0   0.5   1.0  2.0
cluster                       
0        53.3  26.7  18.3  1.7
1        61.1  36.1  

## Step 6 — Summary and save

Compare directly against notebook 15's (buggy, full-Tucker) results.

In [21]:
df_results = pd.DataFrame(all_results)

print(f"Partial Tucker (FIXED)  HOG_rank={HOG_RANK}  IMG_rank={IMG_RANK}  final={tucker_repr.shape} components")
print("=" * 70)
print(df_results.to_string(index=False))

old_path = os.path.join(NB_PADDED, "results_tucker_clustering.csv")
if os.path.exists(old_path):
    df_old = pd.read_csv(old_path)
    print("\nFor comparison, notebook 15's original (buggy, full-Tucker) results:")
    print(df_old.to_string(index=False))

out_path = os.path.join(NB_PADDED, "results_tucker_clustering_fixed.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Partial Tucker (FIXED)  HOG_rank=10  IMG_rank=1  final=(416, 10) components
n_final_components  k  silhouette     ari    nmi  spread_pct_healthy
         (416, 10)  3      0.0679  0.0406 0.0397                32.7
         (416, 10)  4      0.0929 -0.0209 0.0370                10.8

For comparison, notebook 15's original (buggy, full-Tucker) results:
                     method  n_final_components  k  silhouette     ari    nmi  spread_pct_healthy
tucker_(416, 1000, 3)_pca90                  57  3      0.7794 -0.0043 0.0047                42.7
tucker_(416, 1000, 3)_pca90                  57  4      0.7799 -0.0043 0.0047                42.7

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_results_with_padding\results_tucker_clustering_fixed.csv
